In [1]:
import numpy as np

class PCAScratch:
    def __init__(self, n_components):
        self.k = n_components

    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_

        # Full SVD would be wasteful for high-dim data; economy SVD is enough.
        U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

        self.components_ = Vt[: self.k]                 # (k, n_features)
        total_var = np.sum(S ** 2)
        self.explained_variance_ratio_ = (S[: self.k] ** 2) / total_var
        self._U, self._S = U, S
        return self

    def transform(self, X):
        Xc = X - self.mean_
        return Xc @ self.components_.T                  # (n_samples, k)

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


In [2]:
from sklearn.decomposition import PCA as SKPCA

rng = np.random.default_rng(42)
n_samples, n_features, k = 500, 128, 12

latent = rng.normal(size=(n_samples, k))
mixing = rng.normal(size=(k, n_features))
X = latent @ mixing + 0.05 * rng.normal(size=(n_samples, n_features))

mine = PCAScratch(k).fit_transform(X)
sk = SKPCA(n_components=k, svd_solver="full").fit_transform(X)

max_abs_diff_after_sign_align = []
for i in range(k):
    corr = np.corrcoef(mine[:, i], sk[:, i])[0, 1]
    sign = np.sign(corr)
    diff = np.max(np.abs(mine[:, i] * sign - sk[:, i]))
    max_abs_diff_after_sign_align.append(diff)

print("Max abs diff per component (sign-aligned):")
for i, d in enumerate(max_abs_diff_after_sign_align):
    print(f"  PC{i+1}: {d:.10f}")

print(f"\nOverall max abs diff: {max(max_abs_diff_after_sign_align):.10f}")
print("Matches sklearn to 6 decimal places:",
      max(max_abs_diff_after_sign_align) < 1e-6)


Max abs diff per component (sign-aligned):
  PC1: 0.0000000000
  PC2: 0.0000000000
  PC3: 0.0000000000
  PC4: 0.0000000000
  PC5: 0.0000000000
  PC6: 0.0000000000
  PC7: 0.0000000000
  PC8: 0.0000000000
  PC9: 0.0000000000
  PC10: 0.0000000000
  PC11: 0.0000000000
  PC12: 0.0000000000

Overall max abs diff: 0.0000000000
Matches sklearn to 6 decimal places: True


In [3]:
import time
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(7)
n_samples, n_features, n_clusters, k = 4000, 128, 8, 12

centers = rng.normal(scale=4.0, size=(n_clusters, n_features))
labels_true = rng.integers(0, n_clusters, n_samples)
X = centers[labels_true] + rng.normal(scale=1.5, size=(n_samples, n_features))

t0 = time.perf_counter()
km_full = KMeans(n_clusters=n_clusters, n_init=5, random_state=0).fit(X)
t_full = time.perf_counter() - t0
sil_full = silhouette_score(X, km_full.labels_)

pca = PCAScratch(k)
X_reduced = pca.fit_transform(X)

t0 = time.perf_counter()
km_reduced = KMeans(n_clusters=n_clusters, n_init=5, random_state=0).fit(X_reduced)
t_reduced = time.perf_counter() - t0
sil_reduced = silhouette_score(X_reduced, km_reduced.labels_)

n_queries = 20000
queries_full = rng.normal(scale=2.0, size=(n_queries, n_features))
queries_reduced = pca.transform(queries_full)

t0 = time.perf_counter()
km_full.predict(queries_full)
t_infer_full = time.perf_counter() - t0

t0 = time.perf_counter()
km_reduced.predict(queries_reduced)
t_infer_reduced = time.perf_counter() - t0

variance_kept = pca.explained_variance_ratio_.sum() * 100

print(f"Explained variance kept (12/128 dims): {variance_kept:.1f}%")
print(f"Silhouette  full  128-dim: {sil_full:.4f}")
print(f"Silhouette  reduced 12-dim: {sil_reduced:.4f}")
print(f"Silhouette change: {(sil_reduced - sil_full) / sil_full * 100:+.1f}%")
print()
print(f"KMeans fit time   128-dim: {t_full*1000:.1f} ms")
print(f"KMeans fit time    12-dim: {t_reduced*1000:.1f} ms")
print()
print(f"Inference (predict) 128-dim, {n_queries} queries: {t_infer_full*1000:.2f} ms")
print(f"Inference (predict)  12-dim, {n_queries} queries: {t_infer_reduced*1000:.2f} ms")
print(f"Inference speedup: {t_infer_full / t_infer_reduced:.1f}x")


Explained variance kept (12/128 dims): 86.3%
Silhouette  full  128-dim: 0.6068
Silhouette  reduced 12-dim: 0.8639
Silhouette change: +42.4%

KMeans fit time   128-dim: 50.0 ms
KMeans fit time    12-dim: 8.4 ms

Inference (predict) 128-dim, 20000 queries: 3.73 ms
Inference (predict)  12-dim, 20000 queries: 0.90 ms
Inference speedup: 4.1x
